# MESA Benchmark: Full Suite
Bu notebook MESA ve rakiplerinin (DummyRAG, Mem0, Zep, Letta) performanslarını karşılaştırmak için hazırlanmıştır.
VS Code, yerel Jupyter ve Google Colab ortamlarında taşınabilir (portable) olarak çalışır.



In [ ]:
import os
import subprocess
import sys
import time

# 1. Ortam Tespiti ve Taşınabilirlik (Colab Bağımlılıkları Giderildi)
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Colab ortamı tespit edildi. Bağımlılıklar kuruluyor...')
    os.system('apt-get update -qq && apt-get install -y -qq zstd')
    os.system('pip install -q nest_asyncio pyyaml matplotlib')
    os.system('pip install -q letta mem0ai zep_python kuzu chromadb')
    os.system('pip install -e /content/MESA[benchmark]')
    os.system('curl -fsSL https://ollama.com/install.sh | sh')
    os.system('nohup ollama serve > /dev/null 2>&1 &')
    os.chdir('/content')
    if not os.path.exists('MESA'):
        print('MESA reposu bulunamadı, ancak mount edildiği varsayılıyor.')
else:
    print('Yerel ortam tespit edildi. Bağımlılıkların kurulu olduğu varsayılıyor.')

# Asenkron Loop Yaması
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

# Proje Kök Dizinini Bulma (Taşınabilirlik için)
try:
    repo_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
    os.chdir(f"{repo_root}/mesa-benchmark")
except Exception:
    if os.path.exists('/content/MESA/mesa-benchmark'):
        os.chdir('/content/MESA/mesa-benchmark')
    elif os.path.exists('mesa-benchmark'):
        os.chdir('mesa-benchmark')

print(f"Çalışma Dizini: {os.getcwd()}")

# Çevresel değişkenler (Ollama)
os.environ['OPENAI_BASE_URL'] = 'http://127.0.0.1:11434/v1'
os.environ['OPENAI_API_KEY'] = 'ollama-dummy-key'
os.environ['MESA_ZERO_COST_MODE'] = 'true'
os.environ['MESA_LLM_PROVIDER'] = 'ollama'
os.environ['MESA_OLLAMA_URL'] = 'http://127.0.0.1:11434'
os.environ['LLM_MODEL_NAME'] = 'llama3.2:3b'



In [ ]:
import shutil
from pathlib import Path

import yaml

# ── Test Matrisi (Rakipler Dahil) ─────────────────────────────────────────────
TEST_MATRIX = [
    {
        'name': 'mesa_comprehensive',
        'label': 'MESA',
        'client_name': 'mesa_client',
        'adapter_class': 'mesa_benchmark.clients.mesa_client.MesaClientAdapter',
        'dataset_name': 'comprehensive_200',
        'dataset_path': 'mesa_benchmark/datasets/comprehensive_200_dataset.json',
        'dataset_version': 'v2',
        'timeout_ms': 15000,
        'parameters': {'verbose': False},
    },
    {
        'name': 'dummy_comprehensive',
        'label': 'DummyRAG (Baseline)',
        'client_name': 'dummy_client',
        'adapter_class': 'mesa_benchmark.clients.dummy_client.DummyClientAdapter',
        'dataset_name': 'comprehensive_200',
        'dataset_path': 'mesa_benchmark/datasets/comprehensive_200_dataset.json',
        'dataset_version': 'v2',
        'timeout_ms': 10000,
        'parameters': {'verbose': False},
    },
    {
        'name': 'mem0_comprehensive',
        'label': 'Mem0',
        'client_name': 'mem0_client',
        'adapter_class': 'mesa_benchmark.clients.mem0_client.Mem0ClientAdapter',
        'dataset_name': 'comprehensive_200',
        'dataset_path': 'mesa_benchmark/datasets/comprehensive_200_dataset.json',
        'dataset_version': 'v2',
        'timeout_ms': 15000,
        'parameters': {'verbose': False},
    },
    {
        'name': 'zep_comprehensive',
        'label': 'Zep',
        'client_name': 'zep_client',
        'adapter_class': 'mesa_benchmark.clients.zep_client.ZepClientAdapter',
        'dataset_name': 'comprehensive_200',
        'dataset_path': 'mesa_benchmark/datasets/comprehensive_200_dataset.json',
        'dataset_version': 'v2',
        'timeout_ms': 15000,
        'parameters': {'verbose': False},
    },
    {
        'name': 'letta_comprehensive',
        'label': 'Letta',
        'client_name': 'letta_client',
        'adapter_class': 'mesa_benchmark.clients.letta_client.LettaClientAdapter',
        'dataset_name': 'comprehensive_200',
        'dataset_path': 'mesa_benchmark/datasets/comprehensive_200_dataset.json',
        'dataset_version': 'v2',
        'timeout_ms': 15000,
        'parameters': {'verbose': False},
    },
    # Edge Cases (Sadece MESA için detaylı yetenek testleri)
    {
        'name': 'mesa_stress',
        'label': 'MESA — Stress',
        'client_name': 'mesa_client',
        'adapter_class': 'mesa_benchmark.clients.mesa_client.MesaClientAdapter',
        'dataset_name': 'stress_dataset',
        'dataset_path': 'mesa_benchmark/datasets/stress_dataset.json',
        'dataset_version': 'v2',
        'timeout_ms': 20000,
        'parameters': {'verbose': False},
    },
    {
        'name': 'mesa_multi_hop',
        'label': 'MESA — Multi-Hop',
        'client_name': 'mesa_client',
        'adapter_class': 'mesa_benchmark.clients.mesa_client.MesaClientAdapter',
        'dataset_name': 'multi_hop_v1',
        'dataset_path': 'datasets/multi_hop/v1/dataset.json',
        'dataset_version': 'v1',
        'timeout_ms': 10000,
        'parameters': {'verbose': False},
    },
    {
        'name': 'mesa_contradiction',
        'label': 'MESA — Contradiction',
        'client_name': 'mesa_client',
        'adapter_class': 'mesa_benchmark.clients.mesa_client.MesaClientAdapter',
        'dataset_name': 'contradiction_v1',
        'dataset_path': 'datasets/contradiction/v1/dataset.json',
        'dataset_version': 'v1',
        'timeout_ms': 10000,
        'parameters': {'verbose': False},
    }
]

def generate_config(test_def, output_dir='.', iterations=1, seed=42):
    """Generates a config.yaml for a given test definition."""
    config = {
        'suite_name': test_def['label'],
        'iterations': iterations,
        'seed': seed,
        'dataset': {
            'name': test_def['dataset_name'],
            'version': test_def.get('dataset_version', 'v2'),
            'path': test_def['dataset_path'],
            'noise_ratio': 0.0,
        },
        'client': {
            'name': test_def['client_name'],
            'adapter_class': test_def['adapter_class'],
            'timeout_ms': test_def['timeout_ms'],
            'parameters': test_def['parameters'],
        },
        'evaluation': {
            'metrics': ['hit_at_k', 'mrr', 'latency', 'efficiency'],
            'llm_judge_model': 'llama3.2:3b',
            'multi_judge_models': [],
            'enable_agreement': True,
        },
    }
    config_path = os.path.join(output_dir, f'config_{test_def["name"]}.yaml')
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    return config_path



In [ ]:
import sys

# Proje yollarını sys.path'e ekle
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from mesa_benchmark.core.runner import BenchmarkRunner
from mesa_benchmark.metrics.calculator import calculate_metrics_from_jsonl


class FullBenchmarkOrchestrator:
    def __init__(self, test_matrix):
        self.test_matrix = test_matrix
        self.results = {}
        self.start_time = None

    def _clean_state(self):
        for f in Path('.').glob('state.json'):
            f.unlink(missing_ok=True)
        for f in Path('.').glob('.benchmark_state'):
            if f.is_dir():
                shutil.rmtree(f, ignore_errors=True)
        # Clean storages
        for spath in ['/content/mesa_storage', './mesa_storage']:
            if Path(spath).exists():
                shutil.rmtree(spath, ignore_errors=True)

    def run_single_test(self, test_def, seed=42):
        self._clean_state()
        config_path = generate_config(test_def, seed=seed)
        runner = BenchmarkRunner(config_path=config_path)
        runner.run()

        results_file = f'results_{runner.run_id}.jsonl'
        metrics = calculate_metrics_from_jsonl(results_file)
        return metrics.model_dump()

    def run_all(self):
        self.start_time = time.time()
        total = len(self.test_matrix)
        print(f"--- MESA FULL BENCHMARK STARTED ({total} Tests) ---")

        for idx, test_def in enumerate(self.test_matrix, 1):
            name = test_def['name']
            label = test_def['label']
            print(f"[{idx}/{total}] Running {label}...")

            start = time.time()
            try:
                metrics_dict = self.run_single_test(test_def)
                duration = time.time() - start

                self.results[name] = {
                    'label': label,
                    'client': test_def['client_name'],
                    'dataset': test_def['dataset_name'],
                    'metrics': metrics_dict,
                    'duration_s': round(duration, 1),
                    'error': None,
                }
                print(f" ✅ Success ({duration:.0f}s) - Acc: {metrics_dict.get('accuracy', 0)*100:.1f}%")
            except Exception as e:
                duration = time.time() - start
                self.results[name] = {
                    'label': label,
                    'metrics': None,
                    'duration_s': round(duration, 1),
                    'error': str(e)
                }
                print(f" ❌ Failed ({duration:.0f}s): {e}")

        print("--- BENCHMARK COMPLETED ---")
        return self.results

orchestrator = FullBenchmarkOrchestrator(TEST_MATRIX)
all_results = orchestrator.run_all()



In [ ]:
from mesa_benchmark.reports.statistics import (
    compute_run_statistics,
    compute_t_test_p_value,
)

SEEDS = [42, 43, 44, 45, 46]

def run_multi_seed(test_def):
    accuracies = []
    latencies = []
    h5s = []
    for seed in SEEDS:
        try:
            orchestrator._clean_state()
            cfg = generate_config(test_def, seed=seed)
            runner = BenchmarkRunner(config_path=cfg)
            runner.run()
            res_file = f'results_{runner.run_id}.jsonl'
            m = calculate_metrics_from_jsonl(res_file).model_dump()
            accuracies.append(m['accuracy'] * 100)
            latencies.append(m['avg_latency_ms'])
            h5s.append(m['hit_at_5'] * 100)
        except Exception:
            pass
    return accuracies, latencies, h5s

print("\\n📊 MULTI-SEED STATISTICAL ANALYSIS (MESA vs DummyRAG)")
mesa_accs, mesa_lats, mesa_h5s = run_multi_seed(TEST_MATRIX[0]) # MESA
dummy_accs, dummy_lats, dummy_h5s = run_multi_seed(TEST_MATRIX[1]) # DummyRAG

acc_stats = compute_run_statistics(mesa_accs)
h5_stats = compute_run_statistics(mesa_h5s)
lat_stats = compute_run_statistics(mesa_lats)

significance = None
if len(mesa_accs) > 1 and len(dummy_accs) > 1:
    significance = compute_t_test_p_value(mesa_accs, dummy_accs)



## 📊 Benchmark Results



In [ ]:
from IPython.display import Markdown, display

md_lines = [
    '| Test | Accuracy | Hit@1 | Hit@3 | Hit@5 | MRR | Avg Lat. | P95 Lat. |',
    '|:-----|:---------|:------|:------|:------|:----|:---------|:---------|'
]

for name, res in all_results.items():
    if res['error'] or not res['metrics']:
        md_lines.append(f"| **{res['label']}** | ERROR | - | - | - | - | - | - |")
        continue
    m = res['metrics']
    md_lines.append(
        f"| **{res['label']}** | %{m['accuracy']*100:.1f} | %{m['hit_at_1']*100:.1f} | "
        f"%{m['hit_at_3']*100:.1f} | %{m['hit_at_5']*100:.1f} | {m['mrr']:.3f} | "
        f"{m['avg_latency_ms']:.0f}ms | {m['p95_latency_ms']:.0f}ms |"
    )

md_lines.extend([
    '',
    '### 📈 Multi-Seed İstatistiksel Kararlılık (MESA - 5 Seed)',
    '| Metrik | Mean ± Std | 95% CI |',
    '|:-------|:-----------|:-------|',
    f"| **Accuracy** | {acc_stats['formatted_str']}% | ±{acc_stats['ci_95']:.2f} |",
    f"| **Hit@5** | {h5_stats['formatted_str']}% | ±{h5_stats['ci_95']:.2f} |",
    f"| **Latency** | {lat_stats['formatted_str']} ms | ±{lat_stats['ci_95']:.0f} |"
])

if significance:
    sig = '✅ Anlamlı' if significance.get('is_significant') else '⚠️ Anlamsız'
    md_lines.append(f"\\n**MESA vs DummyRAG (Welch t-test):** p={significance.get('p_value_approx', 1.0):.6f} → {sig}")

display(Markdown('\\n'.join(md_lines)))



In [ ]:
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt

# A simple bar chart for Accuracy across main competitors
tests_to_plot = ['mesa_comprehensive', 'dummy_comprehensive', 'mem0_comprehensive', 'zep_comprehensive', 'letta_comprehensive']
labels = []
accs = []
for t in tests_to_plot:
    if t in all_results and all_results[t]['metrics']:
        labels.append(all_results[t]['label'])
        accs.append(all_results[t]['metrics']['accuracy'] * 100)

if accs:
    plt.figure(figsize=(10, 5), facecolor='#1F2937')
    ax = plt.gca()
    ax.set_facecolor('#111827')
    bars = ax.bar(labels, accs, color='#4F46E5', edgecolor='#374151')
    ax.axhline(y=90, color='#10B981', linestyle='--', label='Production Target')
    ax.set_ylabel('Accuracy (%)', color='white')
    ax.tick_params(colors='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"%{bar.get_height():.1f}",
                ha='center', color='white', fontweight='bold')
    plt.title('Competitor Accuracy Comparison (Comprehensive 200)', color='white')
    plt.legend()
    plt.savefig('competitor_accuracy.png', dpi=150, bbox_inches='tight')
    print('✅ Saved competitor_accuracy.png')
